In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from dataclasses import dataclass
from typing import List, Union, Tuple, Callable
import math


In [ ]:
def reynolds_number(v, D, rho, mu):
    return (rho * v * D) / mu
velocities = np.array([0.1,0.5, 1, 1.5, 2, 2.5, 3])
viscosities = np.array([0.0004, 0.0003, 0.0002])
D = 0.3
rho = 980
v = velocities[:, np.newaxis]      
mu = viscosities[np.newaxis, :]    
Re = reynolds_number(v, D, rho, mu)
print(Re)




In [ ]:
#friction factors
def swamee_jain(Re, epsilon, D):
    return 0.25 / ((np.log10((epsilon / (3.7 * D)) + (5.74 / (Re ** 0.9)))) ** 2)
epsilon = 0.000045
friction_factors = swamee_jain(Re, epsilon, D)
print(friction_factors)

PRESSURE LOSS CALCULATION

In [ ]:
@dataclass
class PipeNetwork:
    lengths_m: List[float]
    diameters_m: Union[float, List[float]]
    fractions_out: List[float]     
    rho: float = 980.0
    f: float = 0.013
    minor_loss_factor: float = 
    P0_bar: float = 
    P1_max_bar: float =
    dP_total_set_bar: float = 

    def _diameter_for(self, i: int) -> float:
        if isinstance(self.diameters_m, list):
            return float(self.diameters_m[i])
        return float(self.diameters_m)

    def pressure_profile_bar(self, m_dot0: float) -> Tuple[List[float], List[float], List[float], List[float]]:
        """
        Returns:
          P_bar: pressure at boundaries (len n+1)
          dP_bar: per section (len n)
          m_in: massflow entering each section (len n)
          m_out: massflow leaving each section (after extraction) (len n)
        """
        n = len(self.lengths_m)
        if len(self.fractions_out) != n:
            raise ValueError(f"fractions_out must have length {n} (one per section).")
        # sanity: cumulative removed may not exceed 1
        if sum(self.fractions_out) > 1.0 + 1e-12:
            raise ValueError(f"Sum of fractions_out exceeds 1.0 (={sum(self.fractions_out):.4f}).")

        P = [self.P0_bar]
        dP_sections = []
        m_in_sections = []
        m_out_sections = []

        m0 = max(0.0, float(m_dot0))
        cum_removed = 0.0

        for i in range(n):
            L = float(self.lengths_m[i])
            D = self._diameter_for(i)
            # flow entering this section is what's left from m0 after previous cumulative removals
            m_in = m0 * max(0.0, 1.0 - cum_removed)
            m_in_sections.append(m_in)

            # pressure drop for this section
            if m_in <= 0.0:
                dP_sections.append(0.0)
                P.append(P[-1])
            else:
                A = math.pi * (D**2) / 4.0
                v = m_in / (self.rho * A)

                dP_Pa = self.f * (L / D) * (self.rho * v**2 / 2.0)
                dP_Pa *= self.minor_loss_factor

                dP_bar = dP_Pa / 1e5
                dP_sections.append(dP_bar)
                P.append(P[-1] - dP_bar)

            frac = float(self.fractions_out[i])
            if frac < 0.0 or frac > 1.0:
                raise ValueError(f"fractions_out[{i}] must be in [0, 1]. Got {frac}")

            cum_removed += frac
            if cum_removed > 1.0:
                cum_removed = 1.0

            m_out = m0 * max(0.0, 1.0 - cum_removed)
            m_out_sections.append(m_out)

        return P, dP_sections, m_in_sections, m_out_sections

    def dP_total(self, m_dot0: float) -> float:
        P, _, _, _ = self.pressure_profile_bar(m_dot0)
        return P[0] - P[-1]

    def P_after_sec1(self, m_dot0: float) -> float:
        P, _, _, _ = self.pressure_profile_bar(m_dot0)
        return P[1]


def bisection_root(func: Callable[[float], float], target: float, lo: float = 0.0, hi: float = 500.0,
                  tol: float = 1e-6, max_iter: int = 200) -> float:
    # Solve func(m)=target for monotone increasing func
    def g(x): return func(x) - target

    if g(lo) > 0:
        return lo

    while g(hi) < 0:
        hi *= 2.0
        if hi > 1e7:
            raise RuntimeError("Could not bracket root; check inputs.")

    for _ in range(max_iter):
        mid = 0.5 * (lo + hi)
        if g(mid) <= 0:
            lo = mid
        else:
            hi = mid
        if abs(hi - lo) / max(1.0, lo) < tol:
            break
    return lo
def bisection_crossing_decreasing(func: Callable[[float], float], target: float,
                                 lo: float = 0.0, hi: float = 500.0,
                                 tol: float = 1e-6, max_iter: int = 200) -> float:
    f_lo = func(lo) - target
    if f_lo <= 0:
        return lo  # already satisfies at lo

    # grow hi until it satisfies
    while func(hi) - target > 0:
        hi *= 2.0
        if hi > 1e7:
            raise RuntimeError("Could not bracket crossing for decreasing function; check inputs.")

    for _ in range(max_iter):
        mid = 0.5 * (lo + hi)
        if func(mid) - target > 0:
            lo = mid
        else:
            hi = mid
        if abs(hi - lo) / max(1.0, lo) < tol:
            break

    return hi


def max_flow_both_constraints(net: PipeNetwork, hi: float = 500.0) -> Tuple[float, float, float]:
    # Max flow allowed by total pressure-drop budget
    m_max_from_dP = bisection_root(net.dP_total, target=net.dP_total_set_bar, lo=0.0, hi=hi)

    if net.P_after_sec1(0.0) <= net.P1_max_bar:
        m_min_from_P1 = 0.0
    else:
        m_min_from_P1 = bisection_crossing_decreasing(net.P_after_sec1, target=net.P1_max_bar, lo=0.0, hi=hi)

    if m_min_from_P1 > m_max_from_dP:
        raise RuntimeError(
            f"Infeasible constraints: need m >= {m_min_from_P1:.6f} to get P1 <= {net.P1_max_bar}, "
            f"but ΔP budget allows at most m <= {m_max_from_dP:.6f}."
        )

 
    return m_max_from_dP, m_max_from_dP, m_min_from_P1

def print_table(net: PipeNetwork, m_dot0: float) -> None:
    P, dP_sec, m_in, m_out = net.pressure_profile_bar(m_dot0)
    print("\nSection | L [m]  | D [m]   | m_in [kg/s] | m_out [kg/s] | dP [bar] | P_out [bar] | frac_of_m0")
    print("--------|--------|---------|-------------|--------------|----------|------------|----------")
    n = len(net.lengths_m)
    for i in range(n):
        L = net.lengths_m[i]
        D = net.diameters_m[i] if isinstance(net.diameters_m, list) else net.diameters_m
        frac = net.fractions_out[i]
        print(f"{i+1:>7} | {L:>6.0f} | {D:>7.3f} | {m_in[i]:>11.6f} | {m_out[i]:>12.6f} | "
              f"{dP_sec[i]:>8.4f} | {P[i+1]:>10.4f} | {frac:>8.2f}")

def plot_supply_return_profile_mirrored(
    net: PipeNetwork,
    m_dot0: float,
    deltaP_customer: float = 1.5
) -> None:

    P, dP_sec, _, _ = net.pressure_profile_bar(m_dot0)
    cum_length = [0.0]
    for L in net.lengths_m:
        cum_length.append(cum_length[-1] + L)

    x_supply = np.array(cum_length)
    P_supply = np.array(P)
    dP_rev = dP_sec[::-1]

 
    P_return_start = P_supply[-1] - deltaP_customer
    P_return = [P_return_start]
    for dP in dP_rev:
        P_return.append(P_return[-1] - dP)

    P_return = np.array(P_return)
    x_return = x_supply[::-1]

    # --- Plot ---
    plt.figure(figsize=(10, 5))
    plt.plot(x_supply, P_supply, marker='o', label="Supply")
    plt.plot(x_return, P_return, marker='o', label="Return")

    plt.xlabel("Distance from heat source [m]", fontsize=14)
    plt.ylabel("Pressure [bar]", fontsize=14)
    
    #save plot as pdf
    
    plt.grid(True)
    plt.legend(fontsize=14)
    plt.tight_layout()
    plt.savefig("pressure_profile.pdf", bbox_inches="tight")
    plt.show()

if __name__ == "__main__":
    lengths_m = []
    D_inner_m = 
    fractions_out = [] 

    net = PipeNetwork(
        lengths_m=lengths_m,
        diameters_m=D_inner_m,
        fractions_out=fractions_out,
        rho=980.0,
        f=0.015,
        minor_loss_factor=1.10,
        P0_bar=24.5,
        P1_max_bar=23.0,
        dP_total_set_bar=10
    )

    m_max, m_max_dp, m_min_p1 = max_flow_both_constraints(net, hi=500.0)

    P1 = net.P_after_sec1(m_max)
    dPtot = net.dP_total(m_max)

    print("\n=== FIXED ΔP SIZING (cumulative fractions of m0) ===")
    print(f"P0 fixed            : {net.P0_bar:.3f} bar")
    print(f"ΔP_total set         : {net.dP_total_set_bar:.3f} bar")
    print(f"m_dot0 (max by ΔP)   : {m_max:.6f} kg/s")
    print(f"P after sec1 (P1)    : {P1:.3f} bar  (must be <= {net.P1_max_bar} bar)")
    print(f"ΔP_total at m_dot0   : {dPtot:.3f} bar\n")

    print_table(net, m_max)
    plot_supply_return_profile_mirrored(net, m_max)
    

In [ ]:
if __name__ == "__main__":
    lengths_m = []
    D_inner_m = 
    fractions_out = [] 

    net = PipeNetwork(
        lengths_m=lengths_m,
        diameters_m=D_inner_m,
        fractions_out=fractions_out,
        rho=980.0,
        f=0.015,
        minor_loss_factor=,
        P0_bar=
        P1_max_bar=
        dP_total_set_bar=
    )

    m_max, m_max_dp, m_min_p1 = max_flow_both_constraints(net, hi=500.0)

    P1 = net.P_after_sec1(m_max)
    dPtot = net.dP_total(m_max)

    print("\n=== FIXED ΔP SIZING (cumulative fractions of m0) ===")
    print(f"P0 fixed            : {net.P0_bar:.3f} bar")
    print(f"ΔP_total set         : {net.dP_total_set_bar:.3f} bar")
    print(f"m_dot0 (max by ΔP)   : {m_max:.6f} kg/s")
    print(f"P after sec1 (P1)    : {P1:.3f} bar  (must be <= {net.P1_max_bar} bar)")
    print(f"ΔP_total at m_dot0   : {dPtot:.3f} bar\n")

    print_table(net, m_max)
    plot_supply_return_profile_mirrored(net, m_max)

In [ ]:
maxflow = 3.0*np.pi*((0.3/2)**2) #m^3/s
maxflow_kg_s = maxflow * 980.0 # kg/s, using density of 980 kg/m^3
print(maxflow_kg_s)

In [ ]:

base_fractions_out = np.array([])

lengths_m = []
D_inner_m = 

n_runs = 100
max_change = 0.10
seed = 27

rng = np.random.default_rng(seed)


def generate_random_fractions_fixed_first_two(
    base_fractions,
    max_change=max_change,
    rng=None,
    fixed_indices=(0, 1),
    max_attempts=100_000
):
    """
    Generate random fractions that:
    - keep selected indices fixed, here sections 1 and 2
    - only vary the other fractions by max +/- max_change
    - keep the total sum exactly 1.0
    - keep all fractions non-negative
    """

    if rng is None:
        rng = np.random.default_rng()

    base_fractions = np.array(base_fractions, dtype=float)

    fixed_indices = set(fixed_indices)
    variable_indices = [i for i in range(len(base_fractions)) if i not in fixed_indices]

    for _ in range(max_attempts):
        delta = np.zeros(len(base_fractions))

        # Random perturbation only for variable fractions
        random_delta = rng.uniform(
            -max_change,
            max_change,
            size=len(variable_indices)
        )

       
        random_delta = random_delta - random_delta.mean()

        for idx, d in zip(variable_indices, random_delta):
            delta[idx] = d

        candidate = base_fractions + delta

        # Enforce fixed values exactly
        for idx in fixed_indices:
            candidate[idx] = base_fractions[idx]

        # Check constraints
        if (
            np.all(candidate >= 0)
            and np.all(np.abs(candidate - base_fractions) <= max_change + 1e-12)
            and np.isclose(candidate.sum(), 1.0)
        ):
            return candidate

    raise RuntimeError(
        "Could not generate valid fractions. Try lowering max_change or increasing max_attempts."
    )


# ==============================
# 3. Run simulations
# ==============================
results = []

for run_id in range(n_runs):

    fractions_out_random = generate_random_fractions_fixed_first_two(
        base_fractions=base_fractions_out,
        max_change=max_change,
        rng=rng,
        fixed_indices=(0, 1)
    )

    net = PipeNetwork(
        lengths_m=lengths_m,
        diameters_m=D_inner_m,
        fractions_out=fractions_out_random.tolist(),
        rho=980.0,
        f=0.015,
        minor_loss_factor=,
        P0_bar=,
        P1_max_bar=,
        dP_total_set_bar=
    )

    m_max, m_max_dp, m_min_p1 = max_flow_both_constraints(net, hi=500.0)

    P1 = net.P_after_sec1(m_max)
    dPtot = net.dP_total(m_max)

    row = {
        "run_id": run_id + 1,
        "m_max_kg_s": m_max,
        "m_max_dp_kg_s": m_max_dp,
        "m_min_p1_kg_s": m_min_p1,
        "P1_bar": P1,
        "dP_total_bar": dPtot,
        "fractions_sum": fractions_out_random.sum(),
    }

    # Store each random fraction separately
    for i, frac in enumerate(fractions_out_random, start=1):
        row[f"fraction_out_sec{i}"] = frac

    results.append(row)


results_df = pd.DataFrame(results)

results_df.head()

results_df[[
    "run_id",
    "m_max_kg_s",
    "P1_bar",
    "dP_total_bar",
    "fractions_sum"
]].describe()

plt.figure(figsize=(8, 5))
plt.hist(results_df["m_max_kg_s"], bins=50)
plt.xlabel("Maximum mass flow [kg/s]")
plt.ylabel("Number of simulations")
plt.title("Sensitivity of maximum mass flow to random fractions_out")
plt.grid(True)
plt.show()

lowest = results_df.loc[results_df["m_max_kg_s"].idxmin()]
highest = results_df.loc[results_df["m_max_kg_s"].idxmax()]

print("Lowest m_max:")
print(lowest)

print("\nHighest m_max:")
print(highest)

In [ ]:

base_fractions_out = np.array([])

lengths_m = []
D_inner_m = 

n_runs = 10000
max_change = 0.10
seed = 27

rng = np.random.default_rng(seed)

def generate_random_fractions_fixed_first_two(
    base_fractions,
    max_change=max_change,
    rng=None,
    fixed_indices=(0, 1),
    max_attempts=100_000
):
    """
    Generate random fractions that:
    - keep selected indices fixed, here sections 1 and 2
    - only vary the other fractions by max +/- max_change
    - keep the total sum exactly 1.0
    - keep all fractions non-negative
    """

    if rng is None:
        rng = np.random.default_rng()

    base_fractions = np.array(base_fractions, dtype=float)

    fixed_indices = set(fixed_indices)
    variable_indices = [i for i in range(len(base_fractions)) if i not in fixed_indices]

    for _ in range(max_attempts):
        delta = np.zeros(len(base_fractions))

        # Random perturbation only for variable fractions
        random_delta = rng.uniform(
            -max_change,
            max_change,
            size=len(variable_indices)
        )

   
        random_delta = random_delta - random_delta.mean()

        for idx, d in zip(variable_indices, random_delta):
            delta[idx] = d

        candidate = base_fractions + delta

        # Enforce fixed values exactly
        for idx in fixed_indices:
            candidate[idx] = base_fractions[idx]

        # Check constraints
        if (
            np.all(candidate >= 0)
            and np.all(np.abs(candidate - base_fractions) <= max_change + 1e-12)
            and np.isclose(candidate.sum(), 1.0)
        ):
            return candidate

    raise RuntimeError(
        "Could not generate valid fractions. Try lowering max_change or increasing max_attempts."
    )

base_friction_factor = 0.015
base_minor_loss_factor = 1.10

# Adjustable uncertainty ranges
friction_factor_change_pct = 0.10      # +/- 10%
minor_loss_factor_change_pct = 0.10    # +/- 10%

results = []

for run_id in range(n_runs):

    # Random fractions_out
    fractions_out_random = generate_random_fractions_fixed_first_two(
        base_fractions=base_fractions_out,
        max_change=max_change,
        rng=rng,
        fixed_indices=(0, 1)
    )

    # Random friction factor
    friction_factor_random = base_friction_factor * rng.uniform(
        1 - friction_factor_change_pct,
        1 + friction_factor_change_pct
    )

    # Random minor loss factor
    minor_loss_factor_random = base_minor_loss_factor * rng.uniform(
        1 - minor_loss_factor_change_pct,
        1 + minor_loss_factor_change_pct
    )

    net = PipeNetwork(
        lengths_m=lengths_m,
        diameters_m=D_inner_m,
        fractions_out=fractions_out_random.tolist(),
        rho=980.0,
        f=friction_factor_random,
        minor_loss_factor=minor_loss_factor_random,
        P0_bar=,
        P1_max_bar=,
        dP_total_set_bar=
    )

    m_max, m_max_dp, m_min_p1 = max_flow_both_constraints(net, hi=500.0)

    P1 = net.P_after_sec1(m_max)
    dPtot = net.dP_total(m_max)

    row = {
        "run_id": run_id + 1,
        "m_max_kg_s": m_max,
        "m_max_dp_kg_s": m_max_dp,
        "m_min_p1_kg_s": m_min_p1,
        "P1_bar": P1,
        "dP_total_bar": dPtot,
        "fractions_sum": fractions_out_random.sum(),

        # Store varied hydraulic assumptions
        "friction_factor": friction_factor_random,
        "minor_loss_factor": minor_loss_factor_random,

        # Store deviations from base values
        "friction_factor_change_pct": (
            (friction_factor_random / base_friction_factor) - 1
        ) * 100,
        "minor_loss_factor_change_pct": (
            (minor_loss_factor_random / base_minor_loss_factor) - 1
        ) * 100,
    }

    # Store each random fraction separately
    for i, frac in enumerate(fractions_out_random, start=1):
        row[f"fraction_out_sec{i}"] = frac

    results.append(row)


results_df = pd.DataFrame(results)

results_df.head()

results_df[[
    "run_id",
    "m_max_kg_s",
    "P1_bar",
    "dP_total_bar",
    "fractions_sum",
    "friction_factor",
    "minor_loss_factor",
    "friction_factor_change_pct",
    "minor_loss_factor_change_pct"
]].describe()

fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(results_df["m_max_kg_s"], bins=100)
ax.axvline(x=147, color="red", linestyle="--", linewidth=2, label="Base model: 147 kg/s")

# Add 147 to the x-axis ticks
ticks = list(ax.get_xticks())
ticks.append(147)
ticks = sorted(set(ticks))
ax.set_xticks(ticks)

# Make the 147 tick label red
for tick in ax.get_xticklabels():
    if tick.get_text() == "147":
        tick.set_color("red")

ax.set_xlabel("Maximum mass flow [kg/s]")
ax.set_ylabel("Number of simulations")
ax.grid(True)

fig.savefig("monte_carlo_max_mass_flow.pdf", bbox_inches="tight")
plt.show()

lowest = results_df.loc[results_df["m_max_kg_s"].idxmin()]
highest = results_df.loc[results_df["m_max_kg_s"].idxmax()]

print("Lowest m_max:")
print(lowest)

print("\nHighest m_max:")
print(highest)
plt.figure(figsize=(8, 5))
plt.scatter(results_df["friction_factor"], results_df["m_max_kg_s"], alpha=0.3)
plt.xlabel("Friction factor [-]")
plt.ylabel("Maximum mass flow [kg/s]")
plt.title("Sensitivity of maximum mass flow to friction factor")
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.scatter(results_df["minor_loss_factor"], results_df["m_max_kg_s"], alpha=0.3)
plt.xlabel("Minor loss factor [-]")
plt.ylabel("Maximum mass flow [kg/s]")
plt.title("Sensitivity of maximum mass flow to minor loss factor")
plt.grid(True)
plt.show()